# 首先调用DCF PRG

In [ ]:
import torch
from common.random.PRG import PRG
from config.base_configs import BIT_LEN, LMD, DEVICE, PRG_TYPE, HALF_RING, RING_MAX
from crypto.tensor.RingTensor import RingTensor
from crypto.primitives.function_secret_sharing import DCF_generate, eval, DCF_eval

In [ ]:
def share(tensor: RingTensor, num_of_party: int):
    shares = []
    last_x = tensor.clone()
    for party_id in range(num_of_party - 1):
        r = torch.randint(0, RING_MAX, tensor.shape, dtype=torch.int64)
        x_i = RingTensor(r)
        shares.append(x_i)
        last_x = last_x - x_i
    shares.append(last_x)
    return shares

## 测试21年文章的方案：

In [ ]:
# 定义边界值：
p = torch.tensor(0)
q = torch.tensor(1000)

In [ ]:
r_in = RingTensor.random([1])
print(r_in)

In [82]:
Nsub1 = RingTensor.convert_to_ring(torch.tensor(RING_MAX-1))

In [83]:
gamma = r_in + Nsub1

In [84]:
print(gamma)

[RingTensor
 value:tensor([37696206]) 
 dtype:int 
 scale:1]


In [85]:
k0, k1 = DCF_generate(gamma, RingTensor(torch.tensor(1)))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 23.93it/s]


In [86]:
q1 = (q + 1) % RING_MAX
ap = (p + r_in.tensor) % RING_MAX
aq = (q + r_in.tensor) % RING_MAX
aq1 = (q + 1 + r_in.tensor) % RING_MAX

In [87]:
print("q1:", q1)
print("ap:", ap)
print("aq:", aq)
print("aq1:", aq1)

q1: tensor(1001)
ap: tensor([37696207])
aq: tensor([37697207])
aq1: tensor([37697208])


In [88]:
l1 = (ap > aq) + 0
print(l1)

tensor([0])


In [89]:
l2 = (ap > p) + 0
print(l2)

tensor([1])


In [90]:
l3 = (aq1 > q1) + 0
print(l3)

tensor([1])


In [91]:
l4 = (aq == Nsub1.tensor) + 0
print(l4)

tensor([0])


In [92]:
out = l1 - l2 + l3 + l4
print(out)
out = RingTensor.convert_to_ring(out)

tensor([0])


In [93]:
z_list = share(out, 2)
z0 = z_list[0]
z1 = z_list[1]
print(z0)
print(z1)

[RingTensor
 value:tensor([3637296147]) 
 dtype:int 
 scale:1]
[RingTensor
 value:tensor([657671149]) 
 dtype:int 
 scale:1]


In [113]:
## eval部分
x = torch.tensor(99 + 37696207)
xp = (x + (Nsub1.tensor - p)) % RING_MAX
print(xp)

tensor(37696305)


In [114]:
xq1 = (x + (Nsub1.tensor - q1)) % RING_MAX
print(xq1)

tensor(37695304)


In [115]:
xp_ring = RingTensor.convert_to_ring(xp.unsqueeze(0))
xq1_ring = RingTensor.convert_to_ring(xq1.unsqueeze(0))

In [116]:
print(xp_ring.tensor.flatten())

tensor([37696305])


In [117]:
num_of_value = len(xp_ring.tensor.flatten())
print(num_of_value)

1


In [118]:
SP0,_ = eval(xp_ring,k0,0,2)

In [119]:
SP1,_ = eval(xp_ring,k1,1,2)

In [120]:
SQ0,_ = eval(xq1_ring,k0,0,2)
SQ1,_ = eval(xq1_ring,k1,1,2)

In [121]:
(SP0 + SP1) % RING_MAX

tensor([0])

In [122]:
SQ0 + SQ1

tensor([1])

In [123]:
l5 =  (x > p) + 0
print(l5)

tensor(1)


In [124]:
l6=  (x > q1) + 0
print(l6)

tensor(1)


In [125]:
res0 = 0 - SP0 + SQ0 + z0.tensor

In [126]:
res1 = 1 * (l5 - l6) - SP1 + SQ1 + z1.tensor

In [127]:
print((res0 + res1) % RING_MAX)

tensor([1])
